# 03 — Feature Engineering

**Input:** `data/cleaned_telemetry.parquet`, `data/cleaned_apontamentos.parquet`  
**Output:** `data/features.parquet`

Features construídas por evento de telemetria:
- Contagens de alarmes em janelas: 15min, 30min, 1h, 2h, 4h
- Contagem de alarmes críticos na última 1h
- Tempo desde o último alarme crítico
- Tipo do equipamento (Caminhao=1 / Escavadeira=0)
- Estado operacional atual (via apontamentos)
- Labels: Don't Go nas próximas 1h / 2h / 4h

**Sem data leakage:** todas as features olham apenas para o passado.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import set_seeds
from src.features import build_features

set_seeds(42)
sns.set_theme(style='whitegrid')

DATA_DIR = '../data'

## 1. Carregamento dos Dados Limpos

In [ ]:
# build_features só usa estas 5 colunas — ler apenas elas evita manter as
# outras ~14 colunas (várias categóricas grandes) duplicadas na memória.
NEEDED_COLS = ['TAG', 'timestamp', 'Tipo', 'Id_Criticidade', 'Is_Dont_Go']
df_tele = pd.read_parquet(f'{DATA_DIR}/cleaned_telemetry.parquet', columns=NEEDED_COLS)
df_apon = pd.read_parquet(f'{DATA_DIR}/cleaned_apontamentos.parquet')

print(f'Telemetria: {len(df_tele):,} registros')
print(f'Apontamentos: {len(df_apon):,} registros')

## 2. Construção das Features

In [ ]:
# build_features é vetorizado (rolling + searchsorted, O(n log n) por TAG),
# então roda no dataset completo (ver spec/DECISIONS.md).
print(f'Construindo features para {df_tele["TAG"].nunique()} TAGs ({len(df_tele):,} registros)...')
features = build_features(df_tele, df_apon)
print(f'Features: {features.shape}')
features.head()

## 3. Verificação de Qualidade das Features

In [ ]:
numeric_cols = features.select_dtypes(include='number').columns
nan_counts = features[numeric_cols].isna().sum()
print('NaN em colunas numéricas:')
print(nan_counts[nan_counts > 0] if nan_counts.sum() > 0 else 'Nenhum NaN encontrado.')

for col in ['label_1h', 'label_2h', 'label_4h']:
    rate = features[col].mean() * 100
    print(f'{col}: {rate:.4f}% positivos')

## 4. Distribuição das Features

In [ ]:
count_cols = ['alarm_count_15m', 'alarm_count_30m', 'alarm_count_1h', 'alarm_count_2h', 'alarm_count_4h']
fig, axes = plt.subplots(1, len(count_cols), figsize=(18, 4))
for ax, col in zip(axes, count_cols):
    features[col].hist(bins=30, ax=ax)
    ax.set_title(col)
    ax.set_xlabel('Contagem')
plt.suptitle('Distribuição das Contagens de Alarmes')
plt.tight_layout()
plt.savefig('../data/feature_distributions.png', dpi=150)
plt.show()

## 5. Correlação com os Labels

In [ ]:
feature_cols = count_cols + ['critical_alarm_count_1h', 'time_since_last_critical', 'tipo_caminhao']
corr = features[feature_cols + ['label_1h']].corr()['label_1h'].drop('label_1h').sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
corr.plot(kind='barh', ax=ax)
ax.set_title("Correlação das Features com label_1h (Don't Go em 1h)")
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.savefig('../data/feature_correlation.png', dpi=150)
plt.show()

## 6. Salvando Features

In [ ]:
features.to_parquet(f'{DATA_DIR}/features.parquet', index=False)
print(f'features.parquet salvo: {features.shape}')